# 01 - Consultas directas sobre archivos Parquet (Ejercicio 3)

Se usa DuckDB para explorar los archivos de `data/raw/` **directamente**, sin importarlos a una tabla (`read_parquet`, `glob`, `parquet_schema`).
Cada consulta esta en `sql/01_exploracion/` con su objetivo y fuente; el resultado completo y la decision de cada una se documentan con `python scripts/document_queries.py` (salida: `docs/consultas_resultados.md`).
La explicacion de 3.6 y 3.9 esta en `docs/exploracion.md`.

In [1]:
import os, sys
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60, "display.width", 180, "display.max_colwidth", 60)

# Dentro del contenedor el proyecto esta en /workspace; fuera, en la carpeta padre de notebooks/.
RAIZ = Path("/workspace") if Path("/workspace/sql").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
import taxi_common as tc
DATOS = tc.data_dir().as_posix()          # carpeta data/ segun donde se ejecute

# Ejercicio 3: se consultan los archivos Parquet directamente, SIN base de datos ni tabla (conexion en memoria).
con = duckdb.connect()

def sql(archivo: str) -> pd.DataFrame:
    """Ejecuta un archivo de sql/ y devuelve un DataFrame. La consulta vive en el .sql, no en el notebook."""
    texto = (RAIZ / "sql" / archivo).read_text(encoding="utf-8").replace("/workspace/data", DATOS)
    return con.execute(texto).df()

print("DuckDB", duckdb.__version__, "| datos:", DATOS)

DuckDB 1.5.5 | datos: /workspace/data


## 3.1 Cantidad de archivos

Archivos Parquet por tipo y anio.

Consulta: `sql/01_exploracion/p01_archivos.sql`

In [2]:
sql("01_exploracion/p01_archivos.sql")

,tipo,anio,archivos
0,green,2024,12
1,green,2025,12
2,green,2026,8
3,green,NaN,32
4,yellow,2024,12
5,yellow,2025,12
6,yellow,2026,8
7,yellow,NaN,32
8,NaN,NaN,64


**Interpretacion / decision:** Hay 64 archivos Parquet: 32 de yellow y 32 de green (12 meses en 2024, 12 en 2025 y 8 en 2026). Septiembre a diciembre de 2026 aún no están publicados. Decisión: tratar 2026 como incompleto y comparar siempre el mismo mes entre años.

## 3.2 Cantidad de registros

Registros por tipo y anio de archivo.

Consulta: `sql/01_exploracion/p02_registros.sql`

In [3]:
sql("01_exploracion/p02_registros.sql")

,tipo,anio,registros
0,green,2024,660218
1,green,2025,591375
2,green,2026,337114
3,green,NaN,1588707
4,yellow,2024,41169720
5,yellow,2025,48722602
6,yellow,2026,29703355
7,yellow,NaN,119595677
8,NaN,NaN,121184384


**Interpretacion / decision:** 121,184,384 viajes: yellow 119,595,677 y green 1,588,707 (green es ~1.3 % del total). Coincide con el inventario de `verify_data.py` y con la tabla `trips` (`v08` devuelve 0 filas). Decisión: analizar por tipo de taxi y no promediar yellow y green mezclados.

## 3.3 y 3.4 Columnas y tipos de datos

Esquema de yellow y green, y su comparacion.

Consulta: `sql/01_exploracion/p03_columnas_tipos_yellow.sql`

In [4]:
sql("01_exploracion/p03_columnas_tipos_yellow.sql")

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [5]:
sql("01_exploracion/p04_columnas_tipos_green.sql")

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,lpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,lpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,store_and_fwd_flag,VARCHAR,YES,None,None,None
4,RatecodeID,BIGINT,YES,None,None,None
5,PULocationID,INTEGER,YES,None,None,None
6,DOLocationID,INTEGER,YES,None,None,None
7,passenger_count,BIGINT,YES,None,None,None
8,trip_distance,DOUBLE,YES,None,None,None
9,fare_amount,DOUBLE,YES,None,None,None


In [6]:
sql("01_exploracion/p05_comparar_columnas.sql")

,columna,nombre_yellow,tipo_yellow,nombre_green,tipo_green,situacion
0,cbd_congestion_fee,cbd_congestion_fee,DOUBLE,cbd_congestion_fee,DOUBLE,comun
1,congestion_surcharge,congestion_surcharge,DOUBLE,congestion_surcharge,DOUBLE,comun
2,dolocationid,DOLocationID,INTEGER,DOLocationID,INTEGER,comun
3,extra,extra,DOUBLE,extra,DOUBLE,comun
4,fare_amount,fare_amount,DOUBLE,fare_amount,DOUBLE,comun
5,improvement_surcharge,improvement_surcharge,DOUBLE,improvement_surcharge,DOUBLE,comun
6,mta_tax,mta_tax,DOUBLE,mta_tax,DOUBLE,comun
7,passenger_count,passenger_count,BIGINT,passenger_count,BIGINT,comun
8,payment_type,payment_type,BIGINT,payment_type,BIGINT,comun
9,pulocationid,PULocationID,INTEGER,PULocationID,INTEGER,comun


**Interpretacion / decision:** Yellow tiene 21 columnas y green 22; la comparación lista 25 columnas distintas. Los nombres de fecha difieren (`tpep_*` frente a `lpep_*`); `trip_type` y `ehail_fee` solo existen en green, `airport_fee` solo en yellow y `cbd_congestion_fee` aparece desde 2025. Esto obliga a normalizar a un esquema común (`scripts/taxi_common.py`) y a dejar en NULL lo que no existe.

## Esquema que cambia entre archivos

Columnas con tipo fisico distinto entre archivos, y columnas que no estan en todos los archivos. Condiciona como se combinan los anios.

Consulta: `sql/01_exploracion/p06_cambios_de_tipo_entre_archivos.sql`

In [7]:
sql("01_exploracion/p06_cambios_de_tipo_entre_archivos.sql")

,tipo,columna,tipos_encontrados,archivos_con_la_columna


In [8]:
sql("01_exploracion/p07_columnas_por_archivo.sql")

,tipo,columna,archivos_con_columna,archivos_del_tipo,primer_mes_con_columna
0,green,cbd_congestion_fee,20,32,2025-01
1,green,request_source,3,32,2026-06
2,yellow,cbd_congestion_fee,20,32,2025-01
3,yellow,request_source,3,32,2026-06


**Interpretacion / decision:** `p06` no encontró columnas con tipo físico distinto entre archivos (0 filas). `p07` muestra columnas que no están en todos los archivos: `cbd_congestion_fee` (desde 2025-01, 20 de 32 archivos por tipo) y `request_source` (desde 2026-06, 3 archivos). Por eso se usa `union_by_name=true`, que tolera columnas ausentes. `TRY_CAST` se mantiene por seguridad (hoy no hizo falta) y `request_source` no se incorpora a `trips`.

## 3.5 Muestra de registros

Cinco filas aleatorias por tipo.

Consulta: `sql/01_exploracion/p08_muestra.sql`

In [9]:
sql("01_exploracion/p08_muestra.sql")

,tipo,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,request_source,lpep_pickup_datetime,lpep_dropoff_datetime,ehail_fee,trip_type
0,yellow,2,2024-01-01 01:27:57,2024-01-01 01:41:58,2,3.35,1,N,151,140,1,17.7,1.0,0.5,4.54,0.0,1.0,27.24,2.50,0.0,NaN,None,NaT,NaT,NaN,<NA>
1,yellow,2,2024-01-01 01:56:07,2024-01-01 02:02:04,1,1.26,1,N,137,79,1,8.6,1.0,0.5,2.72,0.0,1.0,16.32,2.50,0.0,NaN,None,NaT,NaT,NaN,<NA>
2,yellow,2,2024-01-01 02:24:34,2024-01-01 03:05:18,1,5.72,1,N,158,236,1,38.7,1.0,0.5,6.00,0.0,1.0,49.70,2.50,0.0,NaN,None,NaT,NaT,NaN,<NA>
3,yellow,2,2024-01-01 03:02:53,2024-01-01 03:12:52,1,2.42,1,N,79,186,2,13.5,1.0,0.5,0.00,0.0,1.0,18.50,2.50,0.0,NaN,None,NaT,NaT,NaN,<NA>
4,yellow,2,2024-01-01 03:07:22,2024-01-01 03:39:17,1,9.58,1,N,181,263,1,41.5,1.0,0.5,9.30,0.0,1.0,55.80,2.50,0.0,NaN,None,NaT,NaT,NaN,<NA>
5,green,2,NaT,NaT,1,0.57,1,N,166,151,1,5.8,1.0,0.5,1.66,0.0,1.0,9.96,0.00,NaN,NaN,None,2024-01-01 20:23:33,2024-01-01 20:26:26,NaN,1
6,green,2,NaT,NaT,1,2.32,1,N,74,263,1,12.1,2.5,0.5,3.77,0.0,1.0,22.62,2.75,NaN,NaN,None,2024-01-02 19:57:51,2024-01-02 20:05:40,NaN,1
7,green,2,NaT,NaT,1,6.86,1,N,116,186,1,31.0,0.0,0.5,7.05,0.0,1.0,42.30,2.75,NaN,NaN,None,2024-01-05 08:13:46,2024-01-05 08:34:25,NaN,1
8,green,2,NaT,NaT,1,2.76,1,N,95,134,2,14.2,1.0,0.5,0.00,0.0,1.0,16.70,0.00,NaN,NaN,None,2024-01-09 23:30:02,2024-01-09 23:47:11,NaN,1
9,green,2,NaT,NaT,1,1.39,1,N,129,260,2,10.0,2.5,0.5,0.00,0.0,1.0,14.00,0.00,NaN,NaN,None,2024-01-12 18:50:51,2024-01-12 18:57:51,NaN,1


## 3.6 Calidad: perfil de cada columna

`SUMMARIZE` entrega minimo, maximo, cuartiles y % de nulos de cada columna.

Consulta: `sql/01_exploracion/p09_resumen_yellow.sql`

In [10]:
sql("01_exploracion/p09_resumen_yellow.sql")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.8335070004244385,0.6161473518861292,2,2,2,119595677,0.00
1,tpep_pickup_datetime,TIMESTAMP,2001-01-01 09:23:58,2026-08-31 23:59:59,59913128,2025-05-15 18:14:36.687007,NaN,2024-10-01 06:57:01.237834,2025-05-23 19:24:27.794099,2025-12-31 08:30:56.958578,119595677,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2001-01-01 16:09:38,2026-09-01 20:16:00,57216129,2025-05-15 18:32:04.653367,NaN,2024-10-01 00:38:45.524905,2025-05-23 17:08:23.512726,2025-12-30 05:16:04.561298,119595677,0.00
3,passenger_count,BIGINT,0,9,11,1.2997810895650606,0.7484946933744885,1,1,1,119595677,19.58
4,trip_distance,DOUBLE,0.0,398608.62,10523,5.879887555051202,554.6138004037873,1.0265604647525386,1.8147252923905215,3.6039413918394168,119595677,0.00
5,RatecodeID,BIGINT,1,99,7,3.174698541566505,14.118714317547855,1,1,1,119595677,19.58
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,119595677,19.58
7,PULocationID,INTEGER,1,265,298,162.46523230935847,65.71491450004375,123,161,233,119595677,0.00
8,DOLocationID,INTEGER,1,265,298,161.89985110414986,70.23822645090647,111,162,234,119595677,0.00
9,payment_type,BIGINT,0,5,6,0.9771172916225057,0.6976295922850996,1,1,1,119595677,0.00


In [11]:
sql("01_exploracion/p10_resumen_green.sql")

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,6,3,2.0460078541858255,0.8552945522798966,2,2,2,1588707,0.00
1,lpep_pickup_datetime,TIMESTAMP,2008-12-31 00:00:00,2026-08-31 23:58:28,1571774,2025-04-04 05:19:26.264852,NaN,2024-08-04 13:46:36.979345,2025-03-21 11:52:28.742311,2025-11-19 22:15:09.16466,1588707,0.00
2,lpep_dropoff_datetime,TIMESTAMP,2008-12-31 00:00:00,2026-09-02 09:39:37,1740542,2025-04-04 05:39:49.432621,NaN,2024-08-04 17:35:25.19127,2025-03-23 09:38:36.713366,2025-11-19 00:50:25.86815,1588707,0.00
3,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,1588707,7.74
4,RatecodeID,BIGINT,1,99,7,1.2536759990284665,1.8724714003995164,1,1,1,1588707,7.74
5,PULocationID,INTEGER,1,265,266,96.73649640871476,56.767371148127935,74,75,103,1588707,0.00
6,DOLocationID,INTEGER,1,265,266,142.1635468339977,76.99613522856896,74,140,228,1588707,0.00
7,passenger_count,BIGINT,0,9,11,1.3039139701608216,0.9577964773685859,1,1,1,1588707,7.74
8,trip_distance,DOUBLE,0.0,262315.94,3609,16.987296688439248,1050.5286543934642,1.1894366808856125,1.9478676935594212,3.4531357494544364,1588707,0.00
9,fare_amount,DOUBLE,-500.0,1676.7,7254,17.97170432936862,17.66704271806289,9.300899848046718,13.572421563109588,20.427503158028703,1588707,0.00


## 3.6 Calidad: conteo de registros problematicos

Fechas, duraciones, distancias, tarifas, pasajeros y zonas invalidas.

Consulta: `sql/01_exploracion/p11_calidad_conteos.sql`

In [12]:
sql("01_exploracion/p11_calidad_conteos.sql")

,tipo,registros,recogida_fuera_del_anio_del_archivo,bajada_antes_de_recogida,duracion_mayor_24h,distancia_cero_o_negativa,distancia_mayor_100_millas,tarifa_cero_o_negativa,total_negativo,pasajeros_nulo_o_cero,zona_recogida_invalida
0,green,1588707,55.0,1351.0,6.0,71224.0,516.0,13015.0,4971.0,142558.0,0.0
1,yellow,119595677,102.0,3820.0,845.0,3131494.0,5706.0,3798269.0,1744900.0,24172589.0,0.0


In [13]:
sql("01_exploracion/p12_fechas_fuera_de_rango.sql")

,tipo,anio_recogida,registros
0,green,2008,10
1,green,2009,8
2,green,2023,2
3,green,2024,660204
4,green,2025,591369
5,green,2026,337114
6,yellow,2001,1
7,yellow,2002,11
8,yellow,2007,1
9,yellow,2008,15


**Problemas de calidad identificados y decision tomada:** Resumen con cifras reales (detalle en `docs/exploracion.md`, sección 3.6): yellow tiene 3.8 M de tarifas <= 0, 3.1 M de distancias <= 0, 1.7 M de totales negativos, 845 viajes de más de 24 h y 24.2 M con pasajeros nulo o 0; hay recogidas con fecha de 2001 a 2009 (68 en yellow, 20 en green fuera de 2024-2026). Decisión: no corregir `trips`; filtrar en cada consulta (distancia 0.1-100 millas, tarifa 1-500, duración 1-180 min y fechas 2024-2026).